In [1]:
#Part 1
#Step 1
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
import math
import time
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
plt.switch_backend('agg')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SOS_token = 0
EOS_token = 1
MAX_LENGTH = 10

In [2]:
#Step 2
class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2 # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
#Step 3
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )


def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()


def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    lines = open(
        'data/%s-%s.txt' % (lang1, lang2),
        encoding='utf-8'
    ).read().strip().split('\n')

    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs


eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)


def filterPair(p):
    return (
        len(p[0].split(' ')) < MAX_LENGTH and
        len(p[1].split(' ')) < MAX_LENGTH and
        p[1].startswith(eng_prefixes)
    )


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]


In [6]:
#Step 4
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])

    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    
    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids
        
    train_data = TensorDataset(torch.LongTensor(input_ids).to(device), torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler,batch_size=batch_size)
    return input_lang, output_lang, train_dataloader
# Build the dataset
batch_size = 32
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counted words:
fra 4601
eng 2991


In [ ]:
#Part 2
#Step 1
class EncoderRNN(nn.Module):
    
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden # output: [B, T, H], hidden: [1, B, H]

In [ ]:
#Step 2
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)


    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):

        batch_size = encoder_outputs.size(0)

        decoder_input = torch.empty(
            batch_size, 1,
            dtype=torch.long,
            device=device
        ).fill_(SOS_token)

        decoder_hidden = encoder_hidden

        decoder_outputs = []

        for i in range(MAX_LENGTH):

            decoder_output, decoder_hidden = self.forward_step(
                decoder_input,
                decoder_hidden
            )

            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)

            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)

        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        return decoder_outputs, decoder_hidden, None
    
    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden

In [ ]:
#Step 3
hidden_size = 128

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_no_attn = DecoderRNN(hidden_size, output_lang.n_words).to(device)

print("Encoder:")
print(encoder)
print(f"\nEncoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")

print("\nDecoder (no attention):")
print(decoder_no_attn)
print(f"\nDecoder parameters: {sum(p.numel() for p in decoder_no_attn.parameters()):,}")

Encoder:
EncoderRNN(
  (embedding): Embedding(4601, 128)
  (gru): GRU(128, 128, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

Encoder parameters: 688,000

Decoder (no attention):
DecoderRNN(
  (embedding): Embedding(2991, 128)
  (gru): GRU(128, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=2991, bias=True)
)

Decoder parameters: 867,759


In [ ]:
#Part 3
#Step 1
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size) # projects decoder state (query)
        self.Ua = nn.Linear(hidden_size, hidden_size) # projects encoder states (keys)
        self.Va = nn.Linear(hidden_size, 1) # reduces to scalar score
        
    def forward(self, query, keys):
        # query: decoder hidden state [batch, 1, hidden_size]
        # keys: encoder outputs [batch, seq_len, hidden_size]
        #
        # Returns:
        # context: weighted sum [batch, 1, hidden_size]
        # weights: attention weights [batch, 1, seq_len]
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        # scores: [batch, seq_len, 1]
        
        scores = scores.squeeze(2).unsqueeze(1)
        # scores: [batch, 1, seq_len]

        weights = F.softmax(scores, dim=-1)
        # weights: [batch, 1, seq_len]

        context = torch.bmm(weights, keys)
        # context: [batch, 1, hidden_size]

        return context, weights

In [12]:
#Step 2
# Create a dummy attention module
attn = BahdanauAttention(hidden_size).to(device)

# Simulate encoder outputs and a decoder hidden state
dummy_encoder_outputs = torch.randn(batch_size, MAX_LENGTH,
hidden_size).to(device)
dummy_decoder_hidden = torch.randn(batch_size, 1, hidden_size).to(device)

context, weights = attn(dummy_decoder_hidden, dummy_encoder_outputs)

print(f"Query shape (decoder hidden): {dummy_decoder_hidden.shape}")
print(f"Keys shape (encoder outputs): {dummy_encoder_outputs.shape}")
print(f"Context vector shape: {context.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Attention weights sum: {weights.sum(dim=-1)}")

Query shape (decoder hidden): torch.Size([32, 1, 128])
Keys shape (encoder outputs): torch.Size([32, 10, 128])
Context vector shape: torch.Size([32, 1, 128])
Attention weights shape: torch.Size([32, 1, 10])
Attention weights sum: tensor([[1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000]], grad_fn=<SumBackward1>)


In [13]:
#Step 2
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)
    
    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long,
        device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []
        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
            decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)
            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()
        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)
        return decoder_outputs, decoder_hidden, attentions
    
    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))
        query = hidden.permute(1, 0, 2) # [1, B, H] -> [B, 1, H]
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2) # [B, 1, 2*H]
        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)
        return output, hidden, attn_weights
    
decoder_attn = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)
print(f"Decoder WITHOUT attention: {sum(p.numel() for p in decoder_no_attn.parameters()):,} params")
print(f"Decoder WITH attention: {sum(p.numel() for p in decoder_attn.parameters()):,} params")
print(f"\nExtra parameters from attention: " f"{sum(p.numel() for p in decoder_attn.parameters()) - sum(p.numel() for p in decoder_no_attn.parameters()):,}")

Decoder WITHOUT attention: 867,759 params
Decoder WITH attention: 950,064 params

Extra parameters from attention: 82,305


In [14]:
#Part 4
#Step 1
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

def train_epoch(dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden,target_tensor)

        loss = criterion(
        decoder_outputs.view(-1, decoder_outputs.size(-1)),
        target_tensor.view(-1)
        )

        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(dataloader)

def showPlot(points, title="Training Loss"):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)
    plt.title(title)
    plt.xlabel("Epoch (x5)")
    plt.ylabel("Loss")
    plt.savefig("training_loss.png")
    plt.show()

In [15]:
#Step 2
def train_model(train_dataloader, encoder, decoder, n_epochs,learning_rate=0.001, print_every=5, plot_every=5):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder,
        encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss
        
        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs), epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0
    showPlot(plot_losses)
    return plot_losses

In [16]:
#Step 3
# Re-initialize encoder and attention decoder
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_attn = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

print("Training Seq2Seq WITH Bahdanau Attention...")
print("=" * 50)
attn_losses = train_model(train_dataloader, encoder, decoder_attn,n_epochs=80, print_every=5, plot_every=5)

Training Seq2Seq WITH Bahdanau Attention...
1m 17s (- 19m 16s) (5 6%) 1.5460
2m 29s (- 17m 26s) (10 12%) 0.6976
3m 42s (- 16m 4s) (15 18%) 0.3662
5m 3s (- 15m 11s) (20 25%) 0.2057
6m 24s (- 14m 5s) (25 31%) 0.1279
7m 43s (- 12m 52s) (30 37%) 0.0886
9m 7s (- 11m 44s) (35 43%) 0.0676
10m 31s (- 10m 31s) (40 50%) 0.0553
11m 54s (- 9m 15s) (45 56%) 0.0473
13m 17s (- 7m 58s) (50 62%) 0.0426
14m 42s (- 6m 41s) (55 68%) 0.0383
16m 4s (- 5m 21s) (60 75%) 0.0360
17m 30s (- 4m 2s) (65 81%) 0.0340
18m 58s (- 2m 42s) (70 87%) 0.0322
20m 21s (- 1m 21s) (75 93%) 0.0310
21m 49s (- 0m 0s) (80 100%) 0.0298


C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\2489845712.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
#Part 5
#Step 1
def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1,-1)

def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        # p.cyan("Encoder outputs:", encoder_outputs)
        # p.yellow("encoder_hidden:", encoder_hidden)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(
        encoder_outputs, encoder_hidden
        )
        # p.yellow("encoder_hidden:", encoder_hidden)
        
        # p.red("Decoder hidden:", decoder_hidden)
        
        # p.green("Decoder outputs:", decoder_outputs)
        # p.blue("Decoder attentions:", decoder_attn)
        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()
        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

def evaluateRandomly(encoder, decoder, pairs, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0],input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

encoder.eval()
decoder_attn.eval()
# Re-load pairs for evaluation
_, _, pairs = prepareData('eng', 'fra', True)
pairs = filterPairs(pairs)
evaluateRandomly(encoder, decoder_attn, pairs)


Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counted words:
fra 4601
eng 2991
> il est content de ses nouvelles chaussures
= he is pleased with his new shoes
< he is pleased with his new shoes <EOS>

> c est un auteur
= he s an author
< he s an ex con <EOS>

> ca me pompe
= i m getting tired of this
< i m getting tired of this bickering <EOS>

> pour moi c est la fin des haricots
= i m at the end of my rope
< i m at the end of my rope <EOS>

> nous entrons en premier
= we re going in first
< we re going in first first <EOS>

> je viens d amerique
= i m from america
< i m from america america <EOS>

> il est dans sa bibliotheque
= he is in his library
< he is in his library <EOS>

> il est plus intelligent qu eux
= he s brighter than they are
< he s brighter than they are <EOS>

> il est lyceen
= he is a student at a high school
< he s a student but a student <EOS>

> ils sont tous en vacances
= they re all on vacation
< they re all on vacation <EOS>



In [18]:
#Step 2
def showAttention(input_sentence, output_words, attentions):
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111)

    attn = attentions.cpu().numpy()

    cax = ax.matshow(attn, cmap='viridis')
    fig.colorbar(cax)

    input_tokens = input_sentence.split(' ') + ['<EOS>']
    output_tokens = output_words

    ax.set_xticks(range(len(input_tokens)))
    ax.set_yticks(range(len(output_tokens)))

    ax.set_xticklabels(input_tokens, rotation=90)
    ax.set_yticklabels(output_tokens)

    plt.xlabel("Source (French)")
    plt.ylabel("Target (English)")
    plt.title("Bahdanau Attention Alignment")

    plt.tight_layout()

    plt.savefig("attention_heatmap.png")
    plt.show()
def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate(
        encoder,
        decoder_attn,
        input_sentence,
        input_lang,
        output_lang
    )

    print('input =', input_sentence)
    print('output =', ' '.join(output_words))

    showAttention(
        input_sentence,
        output_words,
        attentions[0, :len(output_words), :]
    )
    
evaluateAndShowAttention('il n est pas aussi grand que son pere')
evaluateAndShowAttention('je suis trop fatigue pour conduire')
evaluateAndShowAttention('je suis desole si c est une question idiote')
evaluateAndShowAttention('je suis reellement fiere de vous')

input = il n est pas aussi grand que son pere
output = he is not as tall as his father <EOS>


C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


input = je suis trop fatigue pour conduire
output = i m too tired to drive <EOS>


C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


input = je suis desole si c est une question idiote
output = i m sorry if this is a stupid question <EOS>


C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


input = je suis reellement fiere de vous
output = i m really proud of you at home <EOS>


C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
                                            #HW 6
#Task 1

#Створюємо encoder та decoder без механізму уваги
encoder_v = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_v = DecoderRNN(hidden_size, output_lang.n_words).to(device)

print("Training Seq2Seq WITHOUT Attention")

#Навчання vanilla Seq2Seq моделі
#Модель тренується 80 епох для порівняння з attention моделлю
vanilla_losses = train_model(
    train_dataloader,
    encoder_v,
    decoder_v,
    n_epochs=80,
    print_every=5,
    plot_every=5
)

#Evaluation
print("Evaluation WITH Attention")
evaluateRandomly(encoder, decoder_attn, pairs)

print("Evaluation WITHOUT Attention")
evaluateRandomly(encoder_v, decoder_v, pairs)

#Loss comparison
plt.figure()
plt.plot(attn_losses, label="Attention")
plt.plot(vanilla_losses, label="Vanilla")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Comparison")
plt.show()

Training Seq2Seq WITHOUT Attention
0m 56s (- 14m 0s) (5 6%) 1.7088
1m 52s (- 13m 9s) (10 12%) 0.9135
2m 49s (- 12m 15s) (15 18%) 0.5903
3m 45s (- 11m 15s) (20 25%) 0.4012
4m 34s (- 10m 4s) (25 31%) 0.2834
5m 23s (- 8m 59s) (30 37%) 0.2073
6m 12s (- 7m 58s) (35 43%) 0.1555
7m 1s (- 7m 1s) (40 50%) 0.1208
7m 50s (- 6m 6s) (45 56%) 0.0962
8m 39s (- 5m 11s) (50 62%) 0.0793
9m 28s (- 4m 18s) (55 68%) 0.0657
10m 17s (- 3m 25s) (60 75%) 0.0580
11m 6s (- 2m 33s) (65 81%) 0.0508
11m 55s (- 1m 42s) (70 87%) 0.0457
12m 44s (- 0m 50s) (75 93%) 0.0417
13m 33s (- 0m 0s) (80 100%) 0.0395
Evaluation WITH Attention
> je ne suis pas celle que vous pensez
= i m not who you think i am
< i m not who you think i am <EOS>

> ce ne sont pas des criminelles
= they re not criminals
< they re not criminals <EOS>

> tu es fort raffine
= you re very sophisticated
< you re very sophisticated <EOS>

> nous sommes sauvees
= we re saved
< we re saved <EOS>

> il est frais emoulu de l universite
= he is fresh from coll

C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\2489845712.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\1574132710.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
#Task 2
class LuongAttention(nn.Module):

    def __init__(self, hidden_size, method='dot'):
        super(LuongAttention, self).__init__()

        #Тип attention (dot або general)
        self.method = method

        #Для методу general використовується лінійне перетворення
        if method == "general":
            self.W = nn.Linear(hidden_size, hidden_size)

    def score(self, query, keys):

        #query = hidden state декодера
        #keys = encoder outputs

        if self.method == "dot":
            #dot product attention
            #s_i^T * h_j
            return torch.bmm(query, keys.transpose(1,2))

        elif self.method == "general":
            #general attention
            #s_i^T * W * h_j
            query = self.W(query)
            return torch.bmm(query, keys.transpose(1,2))

    def forward(self, query, keys):

        #Обчислення attention scores
        scores = self.score(query, keys)

        #Перетворення scores у attention weights
        weights = F.softmax(scores, dim=-1)

        #Обчислення context vector
        #context = weighted sum encoder outputs
        context = torch.bmm(weights, keys)

        return context, weights


#Тестування реалізації Luong Attention

luong_attn = LuongAttention(hidden_size).to(device)

dummy_encoder_outputs = torch.randn(batch_size, MAX_LENGTH, hidden_size).to(device)
dummy_decoder_hidden = torch.randn(batch_size, 1, hidden_size).to(device)

context, weights = luong_attn(dummy_decoder_hidden, dummy_encoder_outputs)

print("Context shape:", context.shape)
print("Weights shape:", weights.shape)

Context shape: torch.Size([32, 1, 128])
Weights shape: torch.Size([32, 1, 10])


In [ ]:
#Task 3
class BiEncoderRNN(nn.Module):

    def __init__(self, input_size, hidden_size):

        super(BiEncoderRNN, self).__init__()

        self.hidden_size = hidden_size

        #embedding перетворює слова у векторне представлення
        self.embedding = nn.Embedding(input_size, hidden_size)

        #GRU працює у двонаправленому режимі
        self.gru = nn.GRU(
            hidden_size,
            hidden_size,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, x):

        #Перетворення токенів у embeddings
        embedded = self.embedding(x)

        #Передача embeddings у GRU
        outputs, hidden = self.gru(embedded)

        #Після bidirectional GRU отримуємо
        #два hidden states (forward + backward)

        #Їх потрібно об'єднати
        outputs = outputs[:, :, :self.hidden_size] + outputs[:, :, self.hidden_size:]

        return outputs, hidden


#Тестування Bidirectional Encoder

bi_encoder = BiEncoderRNN(input_lang.n_words, hidden_size).to(device)

dummy_input = torch.randint(0, input_lang.n_words, (batch_size, MAX_LENGTH)).to(device)

outputs, hidden = bi_encoder(dummy_input)

print("Encoder outputs shape:", outputs.shape)
print("Hidden shape:", hidden.shape)

Encoder outputs shape: torch.Size([32, 10, 128])
Hidden shape: torch.Size([2, 32, 128])


In [ ]:
#Task 4
import scipy.stats
import numpy as np
import matplotlib.pyplot as plt

entropy_positions = []
entropy_values = []

num_samples = 50

for i in range(num_samples):

    pair = random.choice(pairs)

    output_words, attentions = evaluate(
        encoder,
        decoder_attn,
        pair[0],
        input_lang,
        output_lang
    )

    attentions = attentions.detach().cpu().numpy()

    #attentions shape = [decoder_len, encoder_len]

    for step in range(attentions.shape[0]):

        weights = attentions[step]

        #Робимо weights 1D
        weights = np.ravel(weights)

        #Нормалізація (entropy очікує distribution)
        weights = weights / np.sum(weights)

        #Обчислення entropy
        entropy = scipy.stats.entropy(weights)

        #Гарантуємо float
        entropy = float(np.asarray(entropy).mean())

        entropy_positions.append(step)
        entropy_values.append(entropy)


#Побудова графіку
plt.figure(figsize=(8,5))

plt.scatter(entropy_positions, entropy_values, alpha=0.5)

plt.xlabel("Decoder step")
plt.ylabel("Attention entropy")

plt.title("Attention entropy vs decoder step")

plt.show()

C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\133492255.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [40]:
#Task 5
#Значення для експерименту
teacher_forcing_ratios = [0.0, 0.5, 1.0]

#Словник для збереження результатів
results = {}

for ratio in teacher_forcing_ratios:

    print("======================================")
    print("Training with teacher_forcing_ratio =", ratio)

    #Встановлюємо значення teacher forcing
    teacher_forcing_ratio = ratio

    #Запускаємо навчання моделі
    losses = train_model(
        train_dataloader,
        encoder,
        decoder_attn,
        n_epochs=20
    )

    #Зберігаємо фінальне значення loss
    final_loss = losses[-1]

    results[ratio] = final_loss

    print("Final loss:", final_loss)


#Виводимо результати експерименту
print("\n===== Порівняння результатів =====")

for ratio, loss in results.items():

    print("teacher_forcing_ratio =", ratio, " -> final loss =", loss)

Training with teacher_forcing_ratio = 0.0
1m 14s (- 3m 44s) (5 25%) 0.0205
2m 26s (- 2m 26s) (10 50%) 0.0194
3m 47s (- 1m 15s) (15 75%) 0.0182
4m 59s (- 0m 0s) (20 100%) 0.0192
Final loss: 0.019168154248485197
Training with teacher_forcing_ratio = 0.5


C:\Users\prosi\AppData\Local\Temp\ipykernel_19476\2489845712.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


1m 13s (- 3m 39s) (5 25%) 0.0188
2m 25s (- 2m 25s) (10 50%) 0.0172
3m 37s (- 1m 12s) (15 75%) 0.0166
4m 50s (- 0m 0s) (20 100%) 0.0186
Final loss: 0.018643818870440445
Training with teacher_forcing_ratio = 1.0
1m 12s (- 3m 36s) (5 25%) 0.0174
2m 23s (- 2m 23s) (10 50%) 0.0159
3m 35s (- 1m 11s) (15 75%) 0.0185
4m 47s (- 0m 0s) (20 100%) 0.0149
Final loss: 0.01488678317233158

===== Порівняння результатів =====
teacher_forcing_ratio = 0.0  -> final loss = 0.019168154248485197
teacher_forcing_ratio = 0.5  -> final loss = 0.018643818870440445
teacher_forcing_ratio = 1.0  -> final loss = 0.01488678317233158
